# Compare Query Results

## Setup

In [28]:
import json
import pandas as pd
from IPython.display import HTML, display

queries_df = pd.read_csv("queries.csv")
queries_df = queries_df.set_index("query")

In [29]:
# Disclosure: Used Claude for crafting the table styling and formatting logic below.
def show_results(query, k=5):
    """Display pre-computed BM25 and semantic results from queries.csv side-by-side."""
    row = queries_df.loc[query]
    bm_results = json.loads(row["bm25"])[:k]
    sem_results = json.loads(row["semantic_search"])[:k]

    bm_rows = [
        f"**{r['title']}**<br><small>{r['author'] or '—'} · ⭐ {r['rating']:.1f} · score: {r['score']:.3f}</small>"
        for r in bm_results
    ]
    sem_rows = [
        f"**{r['title']}**<br><small>{r['author'] or '—'} · ⭐ {r['rating']:.1f} · score: {r['score']:.3f}</small>"
        for r in sem_results
    ]

    df = pd.DataFrame({
        "Rank": [f"#{i+1}" for i in range(k)],
        "BM25": bm_rows,
        "Semantic Search": sem_rows,
    }).set_index("Rank")

    display(HTML(df.to_html(escape=False)))

In [30]:
def show_results_LLM(query):
    """Display pre-computed hybrid RAG results from queries.csv."""
    row = queries_df.loc[query]
    hybrid_llama = row["hybrid_rag_llama-3.1-8b-instant"]
    hybrid_gpt_oss = row["hybrid_rag_openai/gpt-oss-20b"]

    print("=== llama-3.1-8b-instant ===")
    print(hybrid_llama)
    print()
    print("=== openai/gpt-oss-20b ===")
    print(hybrid_gpt_oss)

## BM25 vs. Semantic
Milestone 1: Qualitative Evaluation of Retrieval Methods, 4.3 Compare Results

### Query Set

| # | Query | Difficulty | Type |
|---|-------|-----------|------|
| 1 | `harry potter` | Easy | Keyword |
| 2 | `learn python programming beginner` | Easy | Keyword |
| 3 | `overcoming grief after losing a loved one` | Medium | Semantic |
| 4 | `self-help book for managing anxiety and stress at work` | Medium | Semantic |
| 5 | `historical novel about World War 2 from a civilian perspective` | Hard | Multi-attribute |

---

## Query 1 — Easy (Keyword): `harry potter`

In [31]:
show_results("harry potter")

,BM25,Semantic Search
Rank,,
#1,**Harry Potter y el legado maldito / Harry Potter and the Cursed Child (Spanish Edition)**J.K. Rowling · ⭐ 4.5 · score: 20.372,**Fantastic Beasts and Where to Find Them**J.K. Rowling · ⭐ 4.7 · score: 0.958
#2,**كواليس هاري بوتر - Harry Potter Film Wizardry (Arabic Edition)**Brian Sibley · ⭐ 4.1 · score: 20.340,**Harry Potter: The Dark Arts (Tiny Book)**Insight Editions · ⭐ 4.5 · score: 1.025
#3,**Complete 1-st Edition Harry Potter Full Book Set Volumes 1-7 Hardback by J. K. Rowling**J.K. Rowling · ⭐ 4.6 · score: 20.157,**Wizard's Hall**Jane Yolen · ⭐ 4.6 · score: 1.099
#4,**Harry Potter: Exploring Hogwarts ™ Sticky Note Tin Set (Set of 3)**— · ⭐ 4.4 · score: 18.834,**كواليس هاري بوتر - Harry Potter Film Wizardry (Arabic Edition)**Brian Sibley · ⭐ 4.1 · score: 1.135
#5,**Harry Potter: The Dark Arts (Tiny Book)**Insight Editions · ⭐ 4.5 · score: 18.644,**Complete 1-st Edition Harry Potter Full Book Set Volumes 1-7 Hardback by J. K. Rowling**J.K. Rowling · ⭐ 4.6 · score: 1.139


### Analysis

**BM25** correctly retrieves 5 books that specifically mention Harry Potter in the title.

**Semantic search** retrieves 2 books that specifically mention Harry Potter in the title, but also "Fanastic Beasts and Where to Find Them" which is part of the franchise but doesn't specifically mention Harry Potter in the title. "Wizard's Hall" is apparently a different book that came out before Harry Potter and is very similar, so we can see the semantic search finding similar titles here. The Riddle of the New Testament does not seem related though.

**Better: BM25** does better since it retrieves 5 related books, while semantic search only retrieves 3.

---

## Query 2 — Easy (Keyword): `learn python programming beginner`

In [32]:
show_results("learn python programming beginner")

,BM25,Semantic Search
Rank,,
#1,"**LEARN MICROPYTHON: Learn Basics Programming Hardware in MicroPython, ESP32 Programming, Arduino, Example Code, PWM, SPI Bus, I2C Bus and More**— · ⭐ 1.8 · score: 27.686",**101 Extra Python Challenges with Solutions / Code Listings**— · ⭐ 4.5 · score: 0.746
#2,**40 Algorithms Every Programmer Should Know: Hone your problem-solving skills by learning different algorithms and their implementation in Python**Imran Ahmad · ⭐ 4.3 · score: 24.475,"**LEARN MICROPYTHON: Learn Basics Programming Hardware in MicroPython, ESP32 Programming, Arduino, Example Code, PWM, SPI Bus, I2C Bus and More**— · ⭐ 1.8 · score: 0.858"
#3,**101 Extra Python Challenges with Solutions / Code Listings**— · ⭐ 4.5 · score: 23.620,**Guide to Java: A Concise Introduction to Programming (Undergraduate Topics in Computer Science)**James T. Streib · ⭐ 2.4 · score: 1.073
#4,**Visual Basic 6**Harold Davis · ⭐ 3.9 · score: 19.742,**Hands-On Cryptography with Python: Leverage the power of Python to encrypt and decrypt data**— · ⭐ 3.6 · score: 1.103
#5,"**Starting Out with C++: From Control Structures through Objects, Brief Edition plus MyProgrammingLab with Pearson eText - Access Card Package (7th Edition)**Tony Gaddis · ⭐ 4.4 · score: 18.696",**Practical Django Projects (Pratical Projects)**James Bennett · ⭐ 3.8 · score: 1.129


### Analysis

**BM25** finds 3 books that are specifically Python related, though only the first result seems beginner friendly. The last two results seem to be beginner programming related, but no Python related.

**Semantic search** leads with 101 Extra Python Challenges which is not beginner friendly but is Python related. After that is the same as the first result from BM25. The third result is python related but doesn't seem beginner friendly, and the fourth is beginner friendly but it's unclear whether it is specifically Python related. The fifth is related to coding but is for Scratch, not Python, and does not appear beginner friendly.

**Better: BM25** as it's top result is the closest match to the search query. Both algorithms struggled a lot with matching the "beginner" part of the query.

---

## Query 3 — Medium (Semantic): `overcoming grief after losing a loved one`

In [33]:
show_results("overcoming grief after losing a loved one")

,BM25,Semantic Search
Rank,,
#1,**Through It All**— · ⭐ 4.9 · score: 18.774,**Anatomy of Grief: An Inspirational Guide to Surviving the Death of Your Child**Barbara Repczynski · ⭐ 5.0 · score: 0.666
#2,**30 Days toward Healing Your Grief: A Workbook for Healing**— · ⭐ 4.4 · score: 18.145,**I'm Grieving as Fast as I Can: How Young Widows and Widowers Can Cope and Heal**Linda Sones Feinberg · ⭐ 4.5 · score: 0.874
#3,**Gravity's Embrace: A True Unsolved Mystery Surrounding An Alaskan Pilot**— · ⭐ 4.6 · score: 16.932,**Elf-Help Grief Therapy - Self-Help Encouragement 20178-ABBEY**— · ⭐ 4.5 · score: 0.886
#4,**Naked and Not Ashamed**— · ⭐ 4.5 · score: 15.638,**You Will See Your Baby In Heaven: A Man's Perspective of Stillbirth**— · ⭐ 4.9 · score: 0.908
#5,**Small Boy Big Profits: How Resilience And Mindset Can Change Your Life**— · ⭐ 5.0 · score: 15.356,**The Birth We Call Death**Paul H. Dunn · ⭐ 4.7 · score: 0.935


### Analysis

**BM25** does return some grief related books (the 2nd and 4th are clearly grief related) but mostly the results seem unrelated. Interestingly, the 4th result appears the most related based on the title, yet it is only the 4th ranked. (Looking at semantic search, we can see that the same result is ranked 2nd.)

**Semantic search** gets all five results really spot on, returning several books very specifically about grieving after the loss of a loved one such as a child or partner.

**Better: Semantic search** clearly does a better job of encoding the meaning of the search phrase and comes up with very relevant results.

---

## Query 4 — Medium (Semantic): `self-help book for managing anxiety and stress at work`

In [34]:
show_results("self-help book for managing anxiety and stress at work")

,BM25,Semantic Search
Rank,,
#1,**The Art of Calm: Relaxation Through the Five Senses**Brian Luke Seaward · ⭐ 3.6 · score: 24.215,**The Art of Calm: Relaxation Through the Five Senses**Brian Luke Seaward · ⭐ 3.6 · score: 0.888
#2,**Emotional Core Therapy**Robert A Moylan · ⭐ 4.3 · score: 21.699,"**Cognitive Behavioral Therapy: A Psychologist's Guide to Overcome Anxiety, Depression, & Negative Thought Patterns: Psychology Self-Help, Book 5**— · ⭐ 3.8 · score: 0.892"
#3,**The Complete Credit Repair Kit + Cd-rom**Brette Sember · ⭐ 4.1 · score: 20.528,**Emotional Core Therapy**Robert A Moylan · ⭐ 4.3 · score: 0.944
#4,**Loving Someone With Ocd: Help for You & Your Family**Cherry Pedrick · ⭐ 4.5 · score: 20.500,**Anxious for Nothing: Finding Calm in a Chaotic World**— · ⭐ 4.8 · score: 0.973
#5,"**Mandalas and More Adult Coloring Book: Over 100 illustrations for stress relief, Autism, ADHD and depression. For adults and children. By Dr. Robert K. Wheeler Jr. (Fantastic Adult Coloring Books)**Dr. Robert K. Wheeler Jr. · ⭐ 5.0 · score: 19.177",**2019 Daily Self-Care Diary: for BPD**— · ⭐ 3.0 · score: 0.987


### Analysis

**BM25** top two recommendations seem very relevant to managing anxiety and stress based on their titles. I researched Emotional Core Therapy, the 3rd recommendation, which is about stress but appears to be more centered on relationships rather than work specifically. Loving Someone with OCD is not really a self-help book or about stress at work, and the complete credit repair kit is very off the mark. The last result is a coloring book for stress relief, which is related to the "stress" keyword but not much else in the query.

**Semantic search** picks the same top book, but it's recommendations for the 2nd and 3rd rank are much more relevant and clearly self-help style books. The 4th and 5th ranked books are less about stress and anxiety in the workplace, but are self-help type books about mental health in general. The 5th ranked book is the same as the 2nd ranked book in BM25.

**Better: Semantic search** does a better job, producing 3 good top matches while BM25 only produces one good top match.

---

## Query 5 — Hard (Multi-attribute): `historical novel about World War 2 from a civilian perspective`

In [35]:
show_results("historical novel about World War 2 from a civilian perspective")

,BM25,Semantic Search
Rank,,
#1,"**Caves, Cannons and Crinolines**Beverly Stowe McClure · ⭐ 4.3 · score: 20.266",**BBC History Magazine:The Second World War Story**— · ⭐ 5.0 · score: 0.728
#2,"**Just War Reconsidered: Strategy, Ethics, and Theory (Battles and Campaigns Series)**— · ⭐ 4.5 · score: 18.541",**Modern Warfare**Ashley Brown · ⭐ 4.0 · score: 0.742
#3,**A High and Hidden Place: A Novel**Michele Claire Lucas · ⭐ 4.0 · score: 16.172,**The World War Two Reader (Routledge Readers in History)**Gordon Martel · ⭐ 3.7 · score: 0.747
#4,"**Writings of a Rebel Colonel: The Civil War Diary and Letters of Samuel Walkup, 48th North Carolina Infantry**— · ⭐ 3.5 · score: 15.429",**The Meaning of the Second World War (Verso World History Series)**Ernest Mandel · ⭐ 4.2 · score: 0.818
#5,"**A Tale of Two Maidens: A Medieval French Story of Fate, Adventure, and the Hundred Years' War**Anne Echols · ⭐ 4.6 · score: 14.446",**Command decision (Five great classic stories of World War II)**William Wister Haines · ⭐ 4.1 · score: 0.845


### Analysis

**BM25** fails entirely at capturing the World War 2 piece of the query. Its rank 1 result, *Caves, Cannons and Crinolines*, is a Civil War novel, not about World War 2. It likely matched because of "historical," "novel," and "War" in its metadata. Rank 2, Just War Reconsidered, is an academic military ethics textbook. None of the five BM25 results are about world war 2, though they do mostly appear to be historical novels about war. It seems that BM25 captured "historical novel" and "war" from the search term but not World War 2 specifically. Interestingly, very few of the exact search terms appear in the titles of returned books (besides "war"), so it seems that BM25 is needing to go into the descriptions and reviews more to find relevant matches than in previous searches.

**Semantic search** correctly focuses on WWII across all five results but it seems to be pulling mostly academic books rather than novels. The 5th result, Raj and Norah, is a novel and is "a true story of love lost and found in WWII", which appears to actually be a very good match to the query, so it's odd that it is only in 5th place.

**Better: Neither** algorithm gets a very good match. Semantic search manages to capture World War 2 in all of it's results, but not historical novel. BM25 seems to capture the historical novel part of the query and that they are looking for a war theme, but does not return any results about World War 2 in particular. Neither method appears to have captured the "civilian perspective" piece of the search query.

## "llama-3.1-8b-instant" LLM (default) vs. "openai/gpt-oss-20b" LLM
Final Submission, 1.2 Experiment with the LLM

### Query Set
| # | Query | Difficulty | Type |
|---|-------|-----------|------|
| 6 | `stephen king horror novel` | Easy | Keyword |
| 7 | `romance novel with enemies to lovers trope` | Medium | Keyword |
| 8 | `a story about a young person finding their identity` | Medium | Semantic |
| 9 | `mystery thriller with unreliable narrator set in small town` | Hard | Multi-attribute |
| 10 | `best science fiction book for someone who liked The Martian` | Hard | Multi-attribute |

In [36]:
def show_results_LLM(query):
    """Display pre-computed hybrid RAG results from queries.csv."""
    row = queries_df.loc[query]
    hybrid_llama = row["hybrid_rag_llama-3.1-8b-instant"]
    hybrid_gpt_oss = row["hybrid_rag_openai/gpt-oss-20b"]

    print("=== llama-3.1-8b-instant ===")
    print(hybrid_llama)
    print()
    print("=== openai/gpt-oss-20b ===")
    print(hybrid_gpt_oss)

## Query 6 — Easy (Keyword): `stephen king horror novel`

In [37]:
show_results_LLM("stephen king horror novel")

=== llama-3.1-8b-instant ===
Based on the provided Amazon dataset, I recommend the following Stephen King horror novels:

1. **Thinner** (4.5/5) - A #1 national bestseller about a man's quest to reverse his nightmare and find the source of his weight loss.
2. **The Green Mile** (4.8/5) - A character-driven story about a death row supervisor who discovers one of the prisoners may have supernatural healing powers.
3. **FAIRY TALE** (4.4/5) is actually a non-horror book by Stephen King, but I'll exclude it from the list.

These novels are all highly rated and classified as horror by Amazon, making them great recommendations for fans of the genre.

=== openai/gpt-oss-20b ===
**Recommended Stephen King horror novel:**  
**Thinner** – A chilling tale of a man cursed to lose weight at an alarming rate, blending psychological dread with supernatural revenge. The story is widely recognized as one of King’s classic horror works.


## Query 7 — Medium (Keyword): `romance novel with enemies to lovers trope`

In [38]:
show_results_LLM("romance novel with enemies to lovers trope")

=== llama-3.1-8b-instant ===
Based on the Amazon dataset, I recommend the following romance novels with the enemies-to-lovers trope:

1. **Reign Returned: The Felserpent Chronicles** by Katie Keridan - This young adult fantasy romance novel features a complex story with a great romance that brings people together, but ultimately tears them apart. The enemies-to-lovers trope is a significant part of the story, making it a great fit for your request.

2. **Letting Love Win** by Alysia S. Knight - While the description doesn't explicitly mention the enemies-to-lovers trope, the rating and good reviews suggest that it might be a romance novel with a strong focus on relationships, potentially including this trope.

Please note that the description for **Letting Love Win** is quite brief, so it's difficult to make a definitive recommendation. However, based on the available information, it's worth considering.

Unfortunately, I don't have enough information to recommend books from the **For 

## Query 8 — Medium (Semantic): `a story about a young person finding their identity`

In [39]:
show_results_LLM("a story about a young person finding their identity")

=== llama-3.1-8b-instant ===
Based on the context provided, here are some book recommendations that tell a story about a young person finding their identity:

1. **That Guy in Our Women's Studies Class** by Allan D. Hunter: This non-fiction memoir shares the story of a genderqueer individual, Derek, who navigates the complexities of identity and intersectionality in a women's studies class.
2. **Camp** by Lev AC Rosen: This novel follows 16-year-old Randy, who reinvents himself as "Del" to fit into a summer camp for queer teens, exploring the tension between self-acceptance and conformity.
3. **Holding the Light; A Soldier's Journey** by Abbie Combs: Although the blurb doesn't explicitly mention identity, this picture book shares a true story of a young person finding their purpose and overcoming challenges, which can be a powerful metaphor for self-discovery.

These books offer relatable and thought-provoking stories about identity, self-discovery, and growth, which may resonate with 

## Query 9 — Hard (Multi-attribute): `mystery thriller with unreliable narrator set in small town`

In [40]:
show_results_LLM("mystery thriller with unreliable narrator set in small town")

=== llama-3.1-8b-instant ===
Based on the Amazon dataset, I recommend the following books that fit your criteria:

1. **Guilty by Association** by Brad Cooper: This psychological thriller is set in a small town, and its fast-paced plot and unexpected twists make it a great fit for a mystery thriller with an unreliable narrator. (Rating: 4.2)
2. **A Legacy of Lies: A Rock Ridge Mystery**: Although not explicitly stated as an unreliable narrator, the book's complex characters and layered story make it a great fit for fans of mystery thrillers set in small towns. (Rating: 4.3)
3. **Witch & Famous (A Westwick Witches Cozy Mystery)**: This cozy mystery series has a small-town setting and features a cast of complex characters, including witches with their own secrets and motivations. While not necessarily an unreliable narrator, the book's blend of mystery and humor makes it a great fit for fans of the genre. (Rating: 3.9)

These recommendations are based on the provided Amazon dataset and m

## Query 10 — Hard (Multi-attribute): `best science fiction book for someone who liked The Martian`

In [41]:
show_results_LLM("best science fiction book for someone who liked The Martian")

=== llama-3.1-8b-instant ===
Based on the Amazon dataset, I recommend:

1. **Old Man's War** by John Scalzi: This book is a New York Times bestselling science fiction novel that explores themes of space travel, intergalactic politics, and humanity's place in the universe. It has a similar sense of adventure and scope to The Martian, and is the first book in a popular series.

2. **REVOLT OF THE GALAXY** by Stephen Goldin: This book is the 10th installment in the Family d'Alembert series, which features epic space battles, intergalactic politics, and a sense of adventure reminiscent of The Martian. While it's part of a larger series, it can be read as a standalone novel.

3. **Transit to Scorpio** by Alan Burt Akers: This book is a science fiction adventure that follows the story of Dray Prescot and his wife, as they explore the galaxy and face various challenges. It has a similar sense of space travel and exploration to The Martian, and is part of a popular series.

All three of these 

## Key Observations
- both models favor a format with an intro sentence indicating these are recommended based on the amazon dataset, then some kind of list/bullet point format to display the recommended book titles along with a sentence about the book and/or why it matches the query, then a summary sentence.
- the Llama model starts all of it's replies with "Based on the", e.g. "Based on the Amazon dataset, I recommend..." or "ased on the context provided, here are some book recommendations..." while the GPT model has more diverse starting sentences and often favors more of a shorter, title-esque format with bold letters, rather than a sentence.
- the Llama model always lists the recommended books in an ordered list (i.e. 1, 2, 3), while the GPT model lists the books differently in every answer
- the GPT model indicated it had no matching books for the query "mystery thriller with unreliable narrator set in small town", whereas the Llama model recommended three books, despite indicating that the books may not match all of the criteria of the search (i.e. two of the books it indicated they may not have an unreliable narrator).
- similarly, for the "romance novel with enemies to lovers trope", both models returned the same best top match, but the Llama model tried to recommend a second book, even though it noted it it's description that it doesn't mention enemies to lovers, whereas the GPT model only recommended the one good match
- in general, the Llama model recommends more books that the GPT model
- the top recommendations are always identical for the two models
- the GPT model tries to put lots of formatting (e.g. bolding, italics putting in markdown tables, etc) into it's answers, while the Llama model only does ordered lists and emphasizing the book titles.
- for the "best science fiction book for someone who liked The Martian" query, both models do a good job referencing why they chose the books they are recommending specifically in relation to The Martian, but the GPT model is more specific, for example GPT references the "the lone‑survivor vibe of *The Martian*" whereas Llama mentions the more generic "similar sense of space travel and exploration to The Martian".

Overall, it seems that both models agree on the top match from the context (the 5 books provided by hybrid rag search), but the GPT model does a better job of being more concise and only recommending good matches, whereas the Llama model will put the good match first but then continue to recommend more books, even when it notes that they may not include everything the prompt asked for. That being said, the Llama model is much more consistent in it's formatting than GPT. This is likely due to the open nature of the system prompt and the Llama model being smaller and less diverse than the GPT model, so it still returns a consistent format even when that is not explicitly asked for. I hypothesize that GPT would perform consistenly if provided a prompt with more specific guidelines on formatting. 